In [ ]:
pip install faker 

In [ ]:
import random
import uuid
from datetime import datetime, timedelta
from faker import Faker

# Initialize Faker
fake = Faker()

# Let's generate 10 mock users to start with
user_ids = [str(uuid.uuid4())[:8] for _ in range(10)]


def generate_mock_transaction(user_ids):
    # Select a random user
    user = random.choice(user_ids)

    # Generate transaction details
    timestamp = datetime.now() - timedelta(
        minutes=random.randint(1, 1000)
    )  # random time in last 16 hours
    amount = round(random.uniform(10.0, 5000.0), 2)  # Normal transaction range
    location = fake.city()
    device = random.choice(["Mobile_iOS", "Mobile_Android", "Desktop_Web"])

    # --- INJECTING CHAOS (Real-world Data Messiness) ---
    # 1. 5% chance the amount is massive (Potential Fraud Velocity Spike)
    if random.random() < 0.05:
        amount = round(random.uniform(50000.0, 100000.0), 2)

    # 2. 3% chance that the location or device goes missing (Null Values)
    if random.random() < 0.03:
        location = None

    return {
        "transaction_id": str(uuid.uuid4()),
        "user_id": user,
        "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "amount": amount,
        "location": location,
        "device_device": device,
    }


# Generate a batch of 20 sample transactions
sample_batch = [generate_mock_transaction(user_ids) for _ in range(20)]

# Print out the first 3 to see what our raw data looks like
for tx in sample_batch[:3]:
    print(tx)

In [ ]:
import csv
import random
import uuid
from datetime import datetime, timedelta
from faker import Faker

fake = Faker()
user_ids = [str(uuid.uuid4())[:8] for _ in range(10)]


def generate_mock_transaction(user_ids):
    user = random.choice(user_ids)
    timestamp = datetime.now() - timedelta(minutes=random.randint(1, 1000))
    amount = round(random.uniform(10.0, 5000.0), 2)
    location = fake.city()
    device = random.choice(["Mobile_iOS", "Mobile_Android", "Desktop_Web"])

    if random.random() < 0.05:
        amount = round(random.uniform(50000.0, 100000.0), 2)
    if random.random() < 0.03:
        location = None

    return {
        "transaction_id": str(uuid.uuid4()),
        "user_id": user,
        "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "amount": amount,
        "location": location,
        "device_device": device,
    }


# Let's scale it up! Generate 100 transactions this time
sample_batch = [generate_mock_transaction(user_ids) for _ in range(100)]

# Define the name of our output file
output_file = "raw_transactions.csv"

# Define the columns (headers) of our CSV file based on the dictionary keys
csv_columns = [
    "transaction_id",
    "user_id",
    "timestamp",
    "amount",
    "location",
    "device_device",
]

# Write to the CSV file
try:
    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        # Create a writer object that maps dictionaries to rows
        writer = csv.DictWriter(csvfile, fieldnames=csv_columns)

        # Write the header row
        writer.writeheader()

        # Write all the transaction rows
        writer.writerows(sample_batch)

    print(f"🚀 Success! Created '{output_file}' with 100 transaction records.")
except IOError:
    print("❌ Error: An I/O error occurred while writing the file.")

In [ ]:
import csv
import os
import random
import uuid
from datetime import datetime, timedelta
from faker import Faker

fake = Faker()
user_ids = [str(uuid.uuid4())[:8] for _ in range(10)]


def generate_mock_transaction(user_ids):
    user = random.choice(user_ids)
    # Let's say this transaction happened exactly right now
    timestamp = datetime.now()
    amount = round(random.uniform(10.0, 5000.0), 2)
    location = fake.city()
    device = random.choice(["Mobile_iOS", "Mobile_Android", "Desktop_Web"])

    if random.random() < 0.05:
        amount = round(random.uniform(50000.0, 100000.0), 2)
    if random.random() < 0.03:
        location = None

    return {
        "transaction_id": str(uuid.uuid4()),
        "user_id": user,
        "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "amount": amount,
        "location": location,
        "device_device": device,
    }


# Generate 100 fresh transactions
sample_batch = [generate_mock_transaction(user_ids) for _ in range(100)]

# --- MOCK AWS S3 PARTITION LOGIC ---
# Get current date details dynamically
current_time = datetime.now()
year = current_time.strftime("%Y")
month = current_time.strftime("%m")
day = current_time.strftime("%d")

# Create a nested folder path that mimics AWS S3 key prefixes
# It will look like: mock_s3_bucket/year=2026/month=06/day=08/
mock_s3_path = os.path.join(
    "mock_s3_bucket", f"year={year}", f"month={month}", f"day={day}"
)

# Tell Python to physically create these nested folders if they don't exist yet
os.makedirs(mock_s3_path, exist_ok=True)

# Generate a unique file name using a timestamp so files don't overwrite each other
timestamp_str = current_time.strftime("%H%M%S")
output_file_name = f"transactions_{timestamp_str}.csv"
full_destination_path = os.path.join(mock_s3_path, output_file_name)

# Define column headers
csv_columns = [
    "transaction_id",
    "user_id",
    "timestamp",
    "amount",
    "location",
    "device_device",
]

# Write the data into our partitioned "S3" path
with open(full_destination_path, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows(sample_batch)

print(f"📁 Mock Cloud Sync Complete!")
print(f"📍 File successfully landing in storage: {full_destination_path}")

In [ ]:
!pip install mysql-connector-python pandas

In [ ]:
import os
import sqlite3
from datetime import datetime
import pandas as pd


def create_sqlite_db_and_table():
    """Connects to SQLite, wipes the old table if schema was broken, and recreates it."""
    conn = sqlite3.connect("risk_analytics.db")
    cursor = conn.cursor()

    # --- SAFE RESET STEP ---
    # Instead of deleting the file via Windows, we safely drop the table inside SQL!
    cursor.execute("DROP TABLE IF EXISTS customer_transactions;")

    # Re-create the table cleanly
    create_table_query = """
    CREATE TABLE IF NOT EXISTS customer_transactions (
        transaction_id TEXT PRIMARY KEY,
        user_id TEXT NOT NULL,
        transaction_timestamp TEXT NOT NULL,
        amount REAL NOT NULL,
        location TEXT DEFAULT 'UNKNOWN',
        device_device TEXT,
        processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """
    cursor.execute(create_table_query)
    conn.commit()
    cursor.close()
    conn.close()
    print("✅ Local SQLite database & table verified with fresh, clean schema!")


def run_sqlite_etl_pipeline():
    """Extracts data from the mock S3 bucket, transforms it, and loads it into SQLite."""

    current_time = datetime.now()
    year = current_time.strftime("%Y")
    month = current_time.strftime("%m")
    day = current_time.strftime("%d")

    target_folder = os.path.join(
        "mock_s3_bucket", f"year={year}", f"month={month}", f"day={day}"
    )

    if not os.path.exists(target_folder):
        print(f"❌ Error: No partition folder found at {target_folder}")
        return

    files = [f for f in os.listdir(target_folder) if f.endswith(".csv")]
    if not files:
        print("❌ Error: No transaction files found to process.")
        return

    csv_file_path = os.path.join(target_folder, files[0])
    print(f"📦 Extracting data from: {csv_file_path}")

    # 1. EXTRACT
    df = pd.read_csv(csv_file_path)

    # 2. TRANSFORM (Cleaning + Schema Mapping)
    df["location"] = df["location"].fillna("UNKNOWN")

    # Explicitly map the column names perfectly
    df = df.rename(columns={"timestamp": "transaction_timestamp"})
    print("🔄 Transformed records and successfully matched column schemas.")

    # 3. LOAD
    conn = sqlite3.connect("risk_analytics.db")
    try:
        df.to_sql(
            "customer_transactions", conn, if_exists="append", index=False
        )
        conn.commit()

        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM customer_transactions;")
        total_rows = cursor.fetchone()[0]
        print(
            f"🚀 Success! Data warehouse updated. Total records inside database: {total_rows}"
        )
        cursor.close()
    finally:
        # Putting this in a 'finally' block ensures the connection CLOSES
        # even if something else fails later, preventing future file locks!
        conn.close()


if __name__ == "__main__":
    create_sqlite_db_and_table()
    run_sqlite_etl_pipeline()

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("risk_analytics.db")

# --- ADVANCED SQL SECURITY VIEW ---
# This query creates a reusable SQL VIEW that calculates the time gaps between user transactions.
# It uses LAG() to fetch the previous transaction timestamp for each unique user.
fraud_velocity_query = """
SELECT 
    transaction_id,
    user_id,
    transaction_timestamp,
    amount,
    location,
    LAG(transaction_timestamp, 1) OVER (
        PARTITION BY user_id 
        ORDER BY transaction_timestamp
    ) AS previous_transaction_timestamp
FROM customer_transactions
ORDER BY user_id, transaction_timestamp;
"""

# Read the results into pandas to see what the window function did
velocity_df = pd.read_sql_query(fraud_velocity_query, conn)
conn.close()

print("🕵️‍♂️ TRANSACTION VELOCITY RISK ANALYSIS 🕵️‍♂️")
print(velocity_df.head(10))

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("risk_analytics.db")

# This query uses a CTE (the WITH clause) to create a temporary analytical table,
# then queries it to find transactions happening too close together!
risk_scoring_query = """
WITH velocity_calculations AS (
    SELECT 
        transaction_id,
        user_id,
        transaction_timestamp,
        amount,
        location,
        LAG(transaction_timestamp, 1) OVER (
            PARTITION BY user_id 
            ORDER BY transaction_timestamp
        ) AS prev_timestamp
    FROM customer_transactions
)
SELECT 
    transaction_id,
    user_id,
    transaction_timestamp,
    prev_timestamp,
    amount,
    location,
    -- Calculate time difference in seconds
    (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) as seconds_since_last_tx,
    -- Automate a risk flag
    CASE 
        WHEN (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) <= 30 THEN 'CRITICAL_VELOCITY_RISK'
        ELSE 'NORMAL_ACTIVITY'
    END as security_status
FROM velocity_calculations
WHERE prev_timestamp IS NOT NULL
ORDER BY seconds_since_last_tx ASC;
"""

risk_df = pd.read_sql_query(risk_scoring_query, conn)
conn.close()

print("🚨 SECURITY ALERTS CONTROL PANEL 🚨")
print(risk_df.head(10))

In [ ]:
!pip install streamlit

In [ ]:
%%writefile app.py
import sqlite3
import pandas as pd
import streamlit as st

# Configure the web layout
st.set_page_config(page_title="Enterprise Risk Control Tower", layout="wide")
st.title("🚨 Enterprise Risk Control Tower & Velocity Dashboard")
st.markdown("Real-time visibility into high-velocity user risk metrics and financial pipeline health.")

# Pull live insights from our SQLite Data Warehouse
conn = sqlite3.connect("risk_analytics.db")

kpi_query = """
SELECT 
    COUNT(transaction_id) as total_tx, 
    SUM(amount) as total_volume, 
    AVG(amount) as avg_tx_size 
FROM customer_transactions;
"""
kpi_df = pd.read_sql_query(kpi_query, conn)

risk_dashboard_query = """
WITH velocity_calculations AS (
    SELECT 
        transaction_id, user_id, transaction_timestamp, amount, location,
        LAG(transaction_timestamp, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp) AS prev_timestamp
    FROM customer_transactions
)
SELECT 
    user_id, transaction_timestamp, amount, location,
    (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) as seconds_since_last_tx,
    CASE 
        WHEN (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) <= 30 THEN '🚨 CRITICAL_VELOCITY_RISK'
        ELSE '✅ NORMAL'
    END as security_status
FROM velocity_calculations;
"""
main_df = pd.read_sql_query(risk_dashboard_query, conn)
conn.close()

# Render interactive metric blocks
col1, col2, col3 = st.columns(3)
with col1: st.metric(label="Total Transactions Logged", value=f"{kpi_df['total_tx'][0]}")
with col2: st.metric(label="Total Pipeline Volume", value=f"${kpi_df['total_volume'][0]:,.2f}")
with col3: st.metric(label="Average Transaction Size", value=f"${kpi_df['avg_tx_size'][0]:,.2f}")

st.markdown("---")
st.subheader("⚠️ Risk Segmentation Analysis")
st.bar_chart(main_df['security_status'].value_counts())

st.subheader("🕵️ Active Security Signals Ledger")
critical_only = main_df[main_df['security_status'] == '🚨 CRITICAL_VELOCITY_RISK']
st.dataframe(critical_only, use_container_width=True)

In [ ]:
!streamlit run app.py

In [ ]:
import csv
import os
import random
import uuid
from datetime import datetime
from faker import Faker

fake = Faker()

# Keep the user pool small so they overlap heavily
user_ids = [str(uuid.uuid4())[:8] for _ in range(5)]
cities = [
    "Mumbai",
    "Delhi",
    "Bangalore",
    "New York",
    "London",
    "Tokyo",
    "Paris",
]


def generate_geo_chaos_transactions(user_ids, count=150):
    batch = []
    current_time = datetime.now()

    for i in range(count):
        user = random.choice(user_ids)
        # Transactions happening rapid-fire
        timestamp = current_time

        # Most transactions are normal, but let's force some geographic teleportation
        if i % 12 == 0:
            # Same user, same time, but wildly different global cities!
            location = random.choice(["Mumbai", "New York", "Tokyo"])
            amount = round(random.uniform(8000.0, 15000.0), 2)
        else:
            location = random.choice(["Delhi", "Bangalore", "Paris"])
            amount = round(random.uniform(10.0, 1500.0), 2)

        device = random.choice(["Mobile_iOS", "Mobile_Android", "Desktop_Web"])

        batch.append(
            {
                "transaction_id": str(uuid.uuid4()),
                "user_id": user,
                "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
                "amount": amount,
                "location": location,
                "device_device": device,
            }
        )

    return batch


# Generate the chaos batch
geo_batch = generate_geo_chaos_transactions(user_ids)

# Save to our existing partition style
year, month, day = (
    datetime.now().strftime("%Y"),
    datetime.now().strftime("%m"),
    datetime.now().strftime("%d"),
)
target_folder = os.path.join(
    "mock_s3_bucket", f"year={year}", f"month={month}", f"day={day}"
)
os.makedirs(target_folder, exist_ok=True)

full_path = os.path.join(target_folder, "geo_chaos_transactions.csv")
csv_columns = [
    "transaction_id",
    "user_id",
    "timestamp",
    "amount",
    "location",
    "device_device",
]

with open(full_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows(geo_batch)

print(f"💥 Geo-Chaos data landed safely at: {full_path}")

In [ ]:
import os
import sqlite3
from datetime import datetime
import pandas as pd


def run_upgraded_etl_pipeline():
    """Extracts ALL CSV files from today's partition folder, transforms, and loads into SQLite."""

    current_time = datetime.now()
    year = current_time.strftime("%Y")
    month = current_time.strftime("%m")
    day = current_time.strftime("%d")

    target_folder = os.path.join(
        "mock_s3_bucket", f"year={year}", f"month={month}", f"day={day}"
    )

    if not os.path.exists(target_folder):
        print(f"❌ Error: No partition folder found at {target_folder}")
        return

    # Find ALL CSV files inside today's partition folder
    files = [f for f in os.listdir(target_folder) if f.endswith(".csv")]
    if not files:
        print("❌ Error: No transaction files found to process.")
        return

    print(f"📂 Found {len(files)} files in today's partition layer. Processing...")

    conn = sqlite3.connect("risk_analytics.db")

    try:
        # Loop through every single CSV file found
        for file_name in files:
            csv_file_path = os.path.join(target_folder, file_name)
            print(f"📦 Extracting data from: {csv_file_path}")

            # 1. EXTRACT
            df = pd.read_csv(csv_file_path)

            # 2. TRANSFORM
            df["location"] = df["location"].fillna("UNKNOWN")
            df = df.rename(columns={"timestamp": "transaction_timestamp"})

            # 3. LOAD
            # 'INSERT OR IGNORE' prevents duplicate transaction_ids if you run this twice!
            cursor = conn.cursor()
            insert_query = """
            INSERT OR IGNORE INTO customer_transactions 
            (transaction_id, user_id, transaction_timestamp, amount, location, device_device)
            VALUES (?, ?, ?, ?, ?, ?);
            """

            for _, row in df.iterrows():
                cursor.execute(
                    insert_query,
                    (
                        row["transaction_id"],
                        row["user_id"],
                        row["transaction_timestamp"],
                        row["amount"],
                        row["location"],
                        row["device_device"],
                    ),
                )
            conn.commit()
            print(f"✅ Successfully appended rows from {file_name}")

        # Check final total row count inside the warehouse
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM customer_transactions;")
        total_rows = cursor.fetchone()[0]
        print(
            f"\n🚀 SUCCESS! Data Warehouse fully sync'd. New Total Rows: {total_rows}"
        )
        cursor.close()

    finally:
        conn.close()


if __name__ == "__main__":
    run_upgraded_etl_pipeline()

In [ ]:
import sqlite3
import uuid
from datetime import datetime

print("🔄 Step 1: Attempting to open database connection...")
conn = sqlite3.connect("risk_analytics.db")
cursor = conn.cursor()
print("✅ Connected to risk_analytics.db successfully.")

# Create the test fraud target user info
target_fraud_user = "FRAUD_TEST_99"
current_time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 4 raw records happening back-to-back across different continents
test_fraud_rows = [
    (str(uuid.uuid4()), target_fraud_user, current_time_str, 9500.00, "Mumbai", "Mobile_iOS"),
    (str(uuid.uuid4()), target_fraud_user, current_time_str, 12000.50, "New York", "Mobile_Android"),
    (str(uuid.uuid4()), target_fraud_user, current_time_str, 8500.20, "London", "Desktop_Web"),
    (str(uuid.uuid4()), target_fraud_user, current_time_str, 14200.00, "Tokyo", "Mobile_iOS")
]

print("🔄 Step 2: Executing database INSERT statements...")
cursor.executemany("""
    INSERT INTO customer_transactions (transaction_id, user_id, transaction_timestamp, amount, location, device_device)
    VALUES (?, ?, ?, ?, ?, ?);
""", test_fraud_rows)

conn.commit()
print("✅ Data successfully committed to database storage rows.")

print("🔄 Step 3: Verifying warehouse records...")
cursor.execute("SELECT COUNT(*) FROM customer_transactions WHERE user_id = 'FRAUD_TEST_99';")
inserted_count = cursor.fetchone()[0]
cursor.close()
conn.close()

print(f"🎉 SUCCESS! Force-Injected {inserted_count} global teleportation records for: {target_fraud_user}")

In [1]:
import sqlite3
import uuid
from datetime import datetime

conn = sqlite3.connect("risk_analytics.db")
cursor = conn.cursor()

target_fraud_user = "FRAUD_TEST_99"
current_time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

test_fraud_rows = [
    (
        str(uuid.uuid4()),
        target_fraud_user,
        current_time_str,
        9500.00,
        "Mumbai",
        "Mobile_iOS",
    ),
    (
        str(uuid.uuid4()),
        target_fraud_user,
        current_time_str,
        12000.50,
        "New York",
        "Mobile_Android",
    ),
    (
        str(uuid.uuid4()),
        target_fraud_user,
        current_time_str,
        8500.20,
        "London",
        "Desktop_Web",
    ),
    (
        str(uuid.uuid4()),
        target_fraud_user,
        current_time_str,
        14200.00,
        "Tokyo",
        "Mobile_iOS",
    ),
]

cursor.executemany(
    """
    INSERT INTO customer_transactions (transaction_id, user_id, transaction_timestamp, amount, location, device_device)
    VALUES (?, ?, ?, ?, ?, ?);
""",
    test_fraud_rows,
)

conn.commit()
cursor.execute(
    "SELECT COUNT(*) FROM customer_transactions WHERE user_id = 'FRAUD_TEST_99';"
)
inserted_count = cursor.fetchone()[0]
cursor.close()
conn.close()

print(f"🎉 SUCCESS! Force-Injected {inserted_count} records.")

🎉 SUCCESS! Force-Injected 4 records.


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("risk_analytics.db")

impossible_travel_query = """
WITH geo_tracking AS (
    SELECT 
        transaction_id,
        user_id,
        transaction_timestamp,
        location,
        amount,
        -- Peek at previous location for the same user
        LAG(location, 1) OVER (
            PARTITION BY user_id 
            ORDER BY transaction_timestamp, transaction_id
        ) as prev_location,
        -- Peek at previous timestamp for the same user
        LAG(transaction_timestamp, 1) OVER (
            PARTITION BY user_id 
            ORDER BY transaction_timestamp, transaction_id
        ) as prev_timestamp
    FROM customer_transactions
)
SELECT 
    user_id,
    transaction_timestamp,
    location as current_city,
    prev_location as previous_city,
    (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) as seconds_between,
    amount,
    CASE 
        WHEN location <> prev_location AND (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) <= 300 THEN '🚨 IMPOSSIBLE_TRAVEL_FRAUD'
        ELSE '✅ VALID_MOVEMENT'
    END as travel_security_status
FROM geo_tracking
WHERE prev_location IS NOT NULL
  AND travel_security_status = '🚨 IMPOSSIBLE_TRAVEL_FRAUD'
ORDER BY seconds_between ASC;
"""

geo_risk_df = pd.read_sql_query(impossible_travel_query, conn)
conn.close()

print("🌍 IMPOSSIBLE TRAVEL SECURITY LOGS 🌍")
print(geo_risk_df)

🌍 IMPOSSIBLE TRAVEL SECURITY LOGS 🌍
     user_id transaction_timestamp      current_city     previous_city  \
0   13e78c73   2026-06-08 14:02:21  South Donaldberg         Davidtown   
1   13e78c73   2026-06-08 14:02:21  Port Kathrynview  South Donaldberg   
2   13e78c73   2026-06-08 14:02:21  South Teresaport  Port Kathrynview   
3   13e78c73   2026-06-08 14:02:21   Jacquelineville  South Teresaport   
4   13e78c73   2026-06-08 14:02:21           UNKNOWN   Jacquelineville   
..       ...                   ...               ...               ...   
88  ece1a255   2026-06-08 14:02:21          Bellstad     South Melissa   
89  f1c24e94   2026-06-08 14:02:21      Johnsonville   Port Isaiahside   
90  f1c24e94   2026-06-08 14:02:21        East Kevin      Johnsonville   
91  f1c24e94   2026-06-08 14:02:21          Kanestad        East Kevin   
92  f1c24e94   2026-06-08 14:02:21          Rosebury          Kanestad   

    seconds_between   amount     travel_security_status  
0                

In [3]:
%%writefile app.py
import sqlite3
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Enterprise Risk Control Tower", layout="wide")
st.title("🚨 Enterprise Risk Control Tower & Space-Time Velocity Dashboard")
st.markdown("Real-time visibility into high-velocity user risk metrics, impossible travel markers, and financial pipeline health.")

# Connect to live warehouse
conn = sqlite3.connect("risk_analytics.db")

# Query 1: Main KPIs
kpi_query = "SELECT COUNT(transaction_id) as total_tx, SUM(amount) as total_volume, AVG(amount) as avg_tx_size FROM customer_transactions;"
kpi_df = pd.read_sql_query(kpi_query, conn)

# Query 2: Velocity Risk
velocity_query = """
WITH velocity_calculations AS (
    SELECT user_id, transaction_timestamp, amount, location,
        LAG(transaction_timestamp, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp) AS prev_timestamp
    FROM customer_transactions
)
SELECT user_id, transaction_timestamp, amount, location,
    CASE WHEN (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) <= 30 THEN '🚨 CRITICAL_VELOCITY_RISK' ELSE '✅ NORMAL' END as security_status
FROM velocity_calculations;
"""
velocity_df = pd.read_sql_query(velocity_query, conn)

# Query 3: Impossible Travel Risk
impossible_travel_query = """
WITH geo_tracking AS (
    SELECT user_id, transaction_timestamp, location, amount,
        LAG(location, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp, transaction_id) as prev_location,
        LAG(transaction_timestamp, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp, transaction_id) as prev_timestamp
    FROM customer_transactions
)
SELECT user_id, transaction_timestamp, location as current_city, prev_location as previous_city,
    (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) as seconds_between, amount
FROM geo_tracking
WHERE prev_location IS NOT NULL AND location <> prev_location AND (unixepoch(transaction_timestamp) - unixepoch(prev_timestamp)) <= 300;
"""
travel_df = pd.read_sql_query(impossible_travel_query, conn)
conn.close()

# --- DISPLAY KPIs ---
col1, col2, col3 = st.columns(3)
with col1: st.metric(label="Total Transactions Logged", value=f"{kpi_df['total_tx'][0]}")
with col2: st.metric(label="Total Pipeline Volume", value=f"${kpi_df['total_volume'][0]:,.2f}")
with col3: st.metric(label="Average Transaction Size", value=f"${kpi_df['avg_tx_size'][0]:,.2f}")

st.markdown("---")

# --- TWO COLUMN DASHBOARD ---
left_col, right_col = st.columns(2)

with left_col:
    st.subheader("⏱️ Time-Velocity Alerts Log")
    critical_v = velocity_df[velocity_df['security_status'] == '🚨 CRITICAL_VELOCITY_RISK']
    st.dataframe(critical_v, use_container_width=True)

with right_col:
    st.subheader("🌍 Impossible Travel Spatial Log")
    if not travel_df.empty:
        st.dataframe(travel_df, use_container_width=True)
    else:
        st.success("✅ No geographic teleportation anomalies detected in current batch.")

Overwriting app.py


In [5]:
import csv
import os
import random
import uuid
from datetime import datetime, timedelta

# Simulate a pool of 1,000 unique customers making 10,000 total transactions
user_pool = [f"USER_{i:04d}" for i in range(1000)]  # Fixed: swapped 'in' to 'for'
cities = [
    "Mumbai",
    "Delhi",
    "Bangalore",
    "New York",
    "London",
    "Tokyo",
    "Paris",
    "Sydney",
    "Dubai",
]
devices = ["Mobile_iOS", "Mobile_Android", "Desktop_Web"]

print("⚡ Generating 10,000 high-volume enterprise records...")

large_batch = []
start_time = datetime.now() - timedelta(days=2)

for i in range(10000):
    user = random.choice(user_pool)
    # Increment time slightly per transaction
    start_time += timedelta(seconds=random.randint(5, 30))

    # Introduce intentional anomalies
    if i % 50 == 0:  # High amount anomaly
        amount = round(random.uniform(15000.0, 50000.0), 2)
        location = random.choice(["New York", "London", "Tokyo"])
    else:
        amount = round(random.uniform(5.0, 2000.0), 2)
        location = random.choice(cities)

    large_batch.append(
        {
            "transaction_id": str(uuid.uuid4()),
            "user_id": user,
            "timestamp": start_time.strftime("%Y-%m-%d %H:%M:%S"),
            "amount": amount,
            "location": location,
            "device_device": random.choice(devices),
        }
    )

# Save to our partitioned storage
year, month, day = (
    datetime.now().strftime("%Y"),
    datetime.now().strftime("%m"),
    datetime.now().strftime("%d"),
)
target_folder = os.path.join(
    "mock_s3_bucket", f"year={year}", f"month={month}", f"day={day}"
)
os.makedirs(target_folder, exist_ok=True)

full_path = os.path.join(target_folder, "enterprise_10k_transactions.csv")
with open(full_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=large_batch[0].keys())
    writer.writeheader()
    writer.writerows(large_batch)

print(f"📦 10,000 records successfully generated at: {full_path}")

⚡ Generating 10,000 high-volume enterprise records...
📦 10,000 records successfully generated at: mock_s3_bucket\year=2026\month=06\day=08\enterprise_10k_transactions.csv


In [6]:
import sqlite3
import pandas as pd

print("🧹 Re-initializing database warehouse...")
conn = sqlite3.connect("risk_analytics.db")
cursor = conn.cursor()

# Drop existing table to ensure a clean performance benchmark
cursor.execute("DROP TABLE IF EXISTS customer_transactions;")

# Recreate schema
cursor.execute(
    """
    CREATE TABLE customer_transactions (
        transaction_id TEXT PRIMARY KEY,
        user_id TEXT,
        transaction_timestamp TEXT,
        amount REAL,
        location TEXT,
        device_device TEXT
    );
"""
)
conn.commit()

# Load the 10k CSV file
df = pd.read_csv(
    os.path.join(target_folder, "enterprise_10k_transactions.csv")
)
df = df.rename(columns={"timestamp": "transaction_timestamp"})

# Bulk upload to database
df.to_sql("customer_transactions", conn, if_exists="append", index=False)

cursor.execute("SELECT COUNT(*) FROM customer_transactions;")
total_count = cursor.fetchone()[0]
conn.close()

print(f"🚀 Data warehouse completely reloaded with {total_count} rows!")

🧹 Re-initializing database warehouse...
🚀 Data warehouse completely reloaded with 10000 rows!


In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("risk_analytics.db")

# This query transforms chaotic logs into clean ML-ready features
feature_engineering_query = """
WITH velocity_base AS (
    SELECT 
        user_id,
        amount,
        location,
        transaction_timestamp,
        LAG(transaction_timestamp, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp) as prev_time,
        LAG(location, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp) as prev_loc
    FROM customer_transactions
),
risk_flagged_base AS (
    SELECT 
        user_id,
        amount,
        CASE WHEN (unixepoch(transaction_timestamp) - unixepoch(prev_time)) <= 30 THEN 1 ELSE 0 END as velocity_alert,
        CASE WHEN location <> prev_loc AND (unixepoch(transaction_timestamp) - unixepoch(prev_time)) <= 300 THEN 1 ELSE 0 END as travel_alert
    FROM velocity_base
)
SELECT 
    user_id,
    COUNT(*) as total_transaction_count,
    ROUND(SUM(amount), 2) as total_spent,
    ROUND(AVG(amount), 2) as avg_transaction_value,
    ROUND(MAX(amount), 2) as max_single_transaction,
    SUM(velocity_alert) as total_velocity_violations,
    SUM(travel_alert) as total_impossible_travel_violations,
    ROUND(CAST(SUM(velocity_alert) AS REAL) / COUNT(*), 4) * 100 as velocity_risk_percentage
FROM risk_flagged_base
GROUP BY user_id
ORDER BY total_velocity_violations DESC, total_spent DESC;
"""

user_profiles_df = pd.read_sql_query(feature_engineering_query, conn)
conn.close()

print("🎯 ENTERPRISE ML-READY USER RISK PROFILES (Top 10 High-Risk Users) 🎯")
print(user_profiles_df.head(10))

🎯 ENTERPRISE ML-READY USER RISK PROFILES (Top 10 High-Risk Users) 🎯
     user_id  total_transaction_count  total_spent  avg_transaction_value  \
0  USER_0646                       12     80654.23                6721.19   
1  USER_0864                       12     67924.37                5660.36   
2  USER_0509                       17     17506.53                1029.80   
3  USER_0205                       16     15552.22                 972.01   
4  USER_0110                        9      8796.62                 977.40   
5  USER_0037                       10     97159.26                9715.93   
6  USER_0857                        8     89994.72               11249.34   
7  USER_0003                       11     89228.60                8111.69   
8  USER_0455                       14     85050.39                6075.03   
9  USER_0945                       13     84834.62                6525.74   

   max_single_transaction  total_velocity_violations  \
0                35731.70   

In [8]:
%%writefile app.py
import sqlite3
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Enterprise Population Risk Engine", layout="wide")
st.title("🛡️ Enterprise Population Risk Engine (10K Users Scale)")
st.markdown("Macro-level risk behavioral profiling, automated feature engineering, and high-velocity fleet monitoring.")

# Connect to database
conn = sqlite3.connect("risk_analytics.db")

# 1. Pipeline High-Level KPIs
kpi_query = "SELECT COUNT(transaction_id) as total_tx, SUM(amount) as total_volume, AVG(amount) as avg_tx_size FROM customer_transactions;"
kpi_df = pd.read_sql_query(kpi_query, conn)

# 2. Complete Feature Engineered Profiles for Dashboard
feature_query = """
WITH velocity_base AS (
    SELECT user_id, amount, location, transaction_timestamp,
        LAG(transaction_timestamp, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp) as prev_time,
        LAG(location, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp) as prev_loc
    FROM customer_transactions
),
risk_flagged_base AS (
    SELECT user_id, amount,
        CASE WHEN (unixepoch(transaction_timestamp) - unixepoch(prev_time)) <= 30 THEN 1 ELSE 0 END as velocity_alert,
        CASE WHEN location <> prev_loc AND (unixepoch(transaction_timestamp) - unixepoch(prev_time)) <= 300 THEN 1 ELSE 0 END as travel_alert
    FROM velocity_base
)
SELECT user_id, COUNT(*) as total_tx_count, ROUND(SUM(amount), 2) as total_spent,
    ROUND(AVG(amount), 2) as avg_tx_value, ROUND(MAX(amount), 2) as max_single_tx,
    SUM(velocity_alert) as velocity_violations, SUM(travel_alert) as travel_violations,
    ROUND(CAST(SUM(velocity_alert) AS REAL) / COUNT(*), 4) * 100 as velocity_risk_pct
FROM risk_flagged_base
GROUP BY user_id;
"""
profiles_df = pd.read_sql_query(feature_query, conn)
conn.close()

# --- KPI METRIC CARDS ---
col1, col2, col3, col4 = st.columns(4)
with col1: st.metric(label="Total Logged Fleet Transactions", value=f"{kpi_df['total_tx'][0]:,}")
with col2: st.metric(label="Total Monitored Volume", value=f"${kpi_df['total_volume'][0]:,.2f}")
with col3: st.metric(label="Average Ticket Footprint", value=f"${kpi_df['avg_tx_size'][0]:,.2f}")
with col4: 
    total_violations = profiles_df['velocity_violations'].sum() + profiles_df['travel_violations'].sum()
    st.metric(label="Total Pipeline Alerts Triggered", value=f"{total_violations:,}", delta="High Volume", delta_color="inverse")

st.markdown("---")

# --- VISUALIZATIONS SECTION ---
st.subheader("📊 Fleet Anomalies Overview")
viz_col1, viz_col2 = st.columns(2)

with viz_col1:
    st.markdown("**Distribution of Transaction Value Sizes**")
    st.bar_chart(profiles_df.set_index('user_id')['avg_tx_value'].head(50))

with viz_col2:
    st.markdown("**Top Users by Total Capital Spent ($)**")
    st.line_chart(profiles_df.sort_values(by='total_spent', ascending=False).set_index('user_id')['total_spent'].head(30))

st.markdown("---")

# --- ADVANCED RISK SEGMENTATION ---
st.subheader("🎯 Machine Learning-Ready Risk Aggregations")
st.markdown("This live relational matrix maps individual user vectors across the entire 10,000 row fleet, isolating statistical outliers for threat mitigation.")

# Filter to show highest risk users first
high_risk_sorted = profiles_df.sort_values(by=['travel_violations', 'velocity_violations', 'total_spent'], ascending=False)
st.dataframe(high_risk_sorted, use_container_width=True)

Overwriting app.py


In [9]:
%%writefile app.py
import sqlite3
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Enterprise Population Risk Engine", layout="wide")
st.title("🛡️ Enterprise Population Risk Engine (10K Users Scale)")
st.markdown("Macro-level risk behavioral profiling, automated feature engineering, and high-velocity fleet monitoring.")

# Connect to database
conn = sqlite3.connect("risk_analytics.db")

# 1. Pipeline High-Level KPIs
kpi_query = "SELECT COUNT(transaction_id) as total_tx, SUM(amount) as total_volume, AVG(amount) as avg_tx_size FROM customer_transactions;"
kpi_df = pd.read_sql_query(kpi_query, conn)

# 2. Complete Feature Engineered Profiles for Dashboard
feature_query = """
WITH velocity_base AS (
    SELECT user_id, amount, location, transaction_timestamp,
        LAG(transaction_timestamp, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp) as prev_time,
        LAG(location, 1) OVER (PARTITION BY user_id ORDER BY transaction_timestamp) as prev_loc
    FROM customer_transactions
),
risk_flagged_base AS (
    SELECT user_id, amount,
        CASE WHEN (unixepoch(transaction_timestamp) - unixepoch(prev_time)) <= 30 THEN 1 ELSE 0 END as velocity_alert,
        CASE WHEN location <> prev_loc AND (unixepoch(transaction_timestamp) - unixepoch(prev_time)) <= 300 THEN 1 ELSE 0 END as travel_alert
    FROM velocity_base
)
SELECT user_id, COUNT(*) as total_tx_count, ROUND(SUM(amount), 2) as total_spent,
    ROUND(AVG(amount), 2) as avg_tx_value, ROUND(MAX(amount), 2) as max_single_tx,
    SUM(velocity_alert) as velocity_violations, SUM(travel_alert) as travel_violations,
    ROUND(CAST(SUM(velocity_alert) AS REAL) / COUNT(*), 4) * 100 as velocity_risk_pct
FROM risk_flagged_base
GROUP BY user_id;
"""
profiles_df = pd.read_sql_query(feature_query, conn)
conn.close()

# --- KPI METRIC CARDS ---
col1, col2, col3, col4 = st.columns(4)
with col1: st.metric(label="Total Logged Fleet Transactions", value=f"{kpi_df['total_tx'][0]:,}")
with col2: st.metric(label="Total Monitored Volume", value=f"${kpi_df['total_volume'][0]:,.2f}")
with col3: st.metric(label="Average Ticket Footprint", value=f"${kpi_df['avg_tx_size'][0]:,.2f}")
with col4: 
    total_violations = profiles_df['velocity_violations'].sum() + profiles_df['travel_violations'].sum()
    st.metric(label="Total Pipeline Alerts Triggered", value=f"{total_violations:,}", delta="High Volume", delta_color="inverse")

st.markdown("---")

# --- INTERACTIVE INTERFACE: USER LOOKUP ---
st.subheader("🔍 Real-Time Account Security Interrogation")
search_query = st.text_input("Type a User ID to investigate (e.g., USER_0646):", "").strip().upper()

if search_query:
    filtered_user = profiles_df[profiles_df['user_id'] == search_query]
    if not filtered_user.empty:
        st.success(f"🎯 Identity Located for {search_query}")
        
        # Display key insights for searched user
        sc1, sc2, sc3, sc4 = st.columns(4)
        sc1.metric("Total Transactions", f"{filtered_user['total_tx_count'].values[0]}")
        sc2.metric("Total Capital Outflow", f"${filtered_user['total_spent'].values[0]:,.2f}")
        sc3.metric("Velocity Violations", f"{filtered_user['velocity_violations'].values[0]}")
        sc4.metric("Impossible Travel Flags", f"{filtered_user['travel_violations'].values[0]}")
        
        # Warning trigger if malicious threshold is met
        if filtered_user['travel_violations'].values[0] > 0 or filtered_user['velocity_violations'].values[0] > 0:
            st.error("⚠️ ACTION REQUIRED: Account exhibits highly anomalous temporal-spatial markers. Recommend lock-out flag.")
    else:
        st.warning(f"❌ User ID '{search_query}' not found in current 10k fleet logs.")

st.markdown("---")

# --- VISUALIZATIONS SECTION ---
st.subheader("📊 Fleet Anomalies Overview")
viz_col1, viz_col2 = st.columns(2)

with viz_col1:
    st.markdown("**Distribution of Transaction Value Sizes**")
    st.bar_chart(profiles_df.set_index('user_id')['avg_tx_value'].head(50))

with viz_col2:
    st.markdown("**Top Users by Total Capital Spent ($)**")
    st.line_chart(profiles_df.sort_values(by='total_spent', ascending=False).set_index('user_id')['total_spent'].head(30))

st.markdown("---")

# --- ADVANCED RISK SEGMENTATION ---
st.subheader("🎯 Machine Learning-Ready Risk Aggregations")
st.markdown("This live relational matrix maps individual user vectors across the entire 10,000 row fleet, isolating statistical outliers for threat mitigation.")

high_risk_sorted = profiles_df.sort_values(by=['travel_violations', 'velocity_violations', 'total_spent'], ascending=False)
st.dataframe(high_risk_sorted, use_container_width=True)

Overwriting app.py
